In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

class F1PredictionSystem:
    def __init__(self):
        self.driver_skill_model = None
        self.team_skill_model = None
        self.ranking_model = None
        self.scalers = {}
        self.encoders = {}
        self.driver_names = []
        self.team_names = []
        self.circuit_names = []
        self.active_drivers = {}
        self.driver_team_history = {}
        self.training_data = None
        self.circuit_difficulty = {}  # 賽道難度係數
        self.weather_impact = {}      # 天氣影響係數
        self.driver_circuit_performance = {}  # 車手在特定賽道的表現
        self.team_reliability = {}    # 車隊可靠性評分
        
    def prepare_data(self, df):
        print("準備數據中...")
        self.training_data = df.copy()
        
        df_clean = df.copy()
        nan_placeholder = "Unknown_Value"

        # 數據清理
        df_clean['Driver'] = df_clean['Driver'].str.strip().fillna(nan_placeholder)
        df_clean['Team'] = df_clean['Team'].str.strip().fillna(nan_placeholder)
        
        if 'Grand Prix' in df_clean.columns:
            df_clean['Grand Prix'] = df_clean['Grand Prix'].str.strip().fillna(nan_placeholder)

        # 識別活躍車手
        self._identify_active_drivers(df_clean) 
        
        # 建立編碼器
        self.driver_names = sorted(df_clean['Driver'].unique().tolist())
        self.team_names = sorted(df_clean['Team'].unique().tolist())
        
        if 'Grand Prix' in df_clean.columns:
            self.circuit_names = sorted(df_clean['Grand Prix'].unique().tolist())
        else:
            self.circuit_names = ['Monaco', 'Silverstone', 'Monza', 'Spa-Francorchamps', 'Suzuka']

        self.encoders['driver'] = LabelEncoder().fit(self.driver_names)
        self.encoders['team'] = LabelEncoder().fit(self.team_names)
        self.encoders['circuit'] = LabelEncoder().fit(self.circuit_names)
        
        # 數據處理
        df_processed = df_clean.copy() 
        df_processed['driver_encoded'] = self.encoders['driver'].transform(df_processed['Driver'])
        df_processed['team_encoded'] = self.encoders['team'].transform(df_processed['Team'])
        
        if 'Grand Prix' in df_processed.columns:
            df_processed['circuit_encoded'] = self.encoders['circuit'].transform(df_processed['Grand Prix'])
        else:
            df_processed['circuit_encoded'] = np.random.choice(
                len(self.encoders['circuit'].classes_), size=len(df_processed))

        # 計算更精確的特徵
        df_processed = self.create_enhanced_features(df_processed)
        return df_processed

    def _identify_active_drivers(self, df):
        """改進的活躍車手識別"""
        print("識別活躍車手...")
        
        if 'year' not in df.columns:
            df = df.copy()
            df['year'] = range(len(df))
        
        # 計算每個車手的統計數據
        driver_stats = {}
        for driver in df['Driver'].unique():
            driver_data = df[df['Driver'] == driver]
            
            # 計算平均排名和完賽率
            positions = pd.to_numeric(driver_data['Pos'] if 'Pos' in driver_data.columns 
                                    else driver_data.get('Position', []), errors='coerce')
            valid_positions = positions.dropna()
            
            avg_position = valid_positions.mean() if len(valid_positions) > 0 else 15
            finish_rate = len(valid_positions) / len(driver_data) if len(driver_data) > 0 else 0
            recent_races = len(driver_data[driver_data['year'] >= df['year'].max() - 3])
            
            driver_stats[driver] = {
                'avg_position': avg_position,
                'finish_rate': finish_rate,
                'total_races': len(driver_data),
                'recent_races': recent_races,
                'last_year': driver_data['year'].max(),
                'current_team': driver_data.iloc[-1]['Team']
            }
        
        # 選擇活躍車手 - 基於多個因素
        active_drivers = {}
        for driver, stats in driver_stats.items():
            # 活躍度評分
            activity_score = (
                (stats['recent_races'] / 10) * 0.3 +  # 近期參賽頻率
                (min(stats['total_races'] / 50, 1)) * 0.2 +  # 總經驗
                (stats['finish_rate']) * 0.2 +  # 完賽率
                (max(0, 20 - stats['avg_position']) / 20) * 0.3  # 平均表現
            )
            
            if activity_score > 0.3 or stats['recent_races'] > 5:
                active_drivers[driver] = {
                    'current_team': stats['current_team'],
                    'last_active_year': stats['last_year'],
                    'total_races': stats['total_races'],
                    'activity_score': activity_score,
                    'avg_position': stats['avg_position'],
                    'finish_rate': stats['finish_rate']
                }
        
        # 至少保留前25名車手
        if len(active_drivers) < 25:
            sorted_drivers = sorted(driver_stats.items(), 
                                  key=lambda x: (x[1]['recent_races'], x[1]['total_races']), 
                                  reverse=True)
            for driver, stats in sorted_drivers[:25]:
                if driver not in active_drivers:
                    active_drivers[driver] = {
                        'current_team': stats['current_team'],
                        'last_active_year': stats['last_year'],
                        'total_races': stats['total_races'],
                        'activity_score': 0.3,
                        'avg_position': stats['avg_position'],
                        'finish_rate': stats['finish_rate']
                    }
        
        self.active_drivers = active_drivers
        print(f"識別出 {len(active_drivers)} 位活躍車手")

    def create_enhanced_features(self, df):
        """創建增強的特徵工程"""
        print("計算增強特徵...")
        
        df = df.sort_values(['year', 'Driver']).reset_index(drop=True)
        
        # 初始化特徵
        df['driver_skill'] = 50.0
        df['team_skill'] = 50.0
        df['driver_consistency'] = 50.0
        df['team_reliability'] = 50.0
        df['driver_circuit_expertise'] = 50.0
        df['recent_form'] = 50.0
        
        # 處理位置數據
        if 'Pos' in df.columns:
            df['position_numeric'] = pd.to_numeric(df['Pos'], errors='coerce')
        elif 'Position' in df.columns:
            df['position_numeric'] = pd.to_numeric(df['Position'], errors='coerce')
        else:
            df['position_numeric'] = np.random.randint(1, 21, len(df))
        
        # 處理DNF等情況
        df['did_finish'] = df['position_numeric'].notna()
        df['position_numeric'].fillna(21, inplace=True)  # DNF給21位
        
        # 計算位置分數（反向，1位=100分，20位=5分）
        df['position_score'] = np.where(df['did_finish'], 
                                       105 - df['position_numeric'] * 5,
                                       np.maximum(20 - df['position_numeric'], 0))
        
        # 計算車手特徵
        print("計算車手特徵...")
        for driver in df['Driver'].unique():
            driver_mask = df['Driver'] == driver
            driver_data = df[driver_mask].sort_values('year')
            
            if len(driver_data) > 1:
                # 車手技能 - 使用指數移動平均
                ema_alpha = 0.3
                driver_skill_ema = driver_data['position_score'].ewm(alpha=ema_alpha).mean()
                df.loc[driver_mask, 'driver_skill'] = driver_skill_ema
                
                # 一致性 - 基於成績方差
                rolling_std = driver_data['position_score'].rolling(window=5, min_periods=1).std()
                consistency = 100 - np.minimum(rolling_std * 2, 50)  # 標準差越小一致性越高
                df.loc[driver_mask, 'driver_consistency'] = consistency.fillna(50)
                
                # 近期狀態 - 最近5場比賽的加權平均
                recent_window = min(5, len(driver_data))
                if recent_window > 0:
                    recent_scores = driver_data['position_score'].tail(recent_window)
                    weights = np.exp(np.linspace(-1, 0, recent_window))  # 指數權重
                    recent_form = np.average(recent_scores, weights=weights)
                    df.loc[driver_mask, 'recent_form'] = recent_form
                
                # 車手在特定賽道的專長
                for circuit in driver_data['circuit_encoded'].unique():
                    circuit_mask = driver_mask & (df['circuit_encoded'] == circuit)
                    circuit_data = df[circuit_mask]['position_score']
                    if len(circuit_data) > 0:
                        circuit_avg = circuit_data.mean()
                        df.loc[circuit_mask, 'driver_circuit_expertise'] = circuit_avg
        
        # 計算車隊特徵
        print("計算車隊特徵...")
        for team in df['Team'].unique():
            team_mask = df['Team'] == team
            team_data = df[team_mask].sort_values('year')
            
            if len(team_data) > 1:
                # 車隊實力 - 按年度計算平均表現
                yearly_performance = team_data.groupby('year')['position_score'].agg(['mean', 'count']).reset_index()
                yearly_performance['weighted_mean'] = (yearly_performance['mean'] * 
                                                     np.sqrt(yearly_performance['count']))  # 樣本量權重
                
                # 車隊實力使用指數移動平均
                team_ema = yearly_performance['weighted_mean'].ewm(alpha=0.25).mean()
                
                for i, year in enumerate(yearly_performance['year']):
                    year_mask = team_mask & (df['year'] == year)
                    df.loc[year_mask, 'team_skill'] = team_ema.iloc[i]
                
                # 車隊可靠性 - 基於完賽率和一致性
                finish_rates = team_data.groupby('year')['did_finish'].mean()
                reliability_ema = finish_rates.ewm(alpha=0.3).mean() * 100
                
                for i, year in enumerate(yearly_performance['year']):
                    year_mask = team_mask & (df['year'] == year)
                    if i < len(reliability_ema):
                        df.loc[year_mask, 'team_reliability'] = reliability_ema.iloc[i]
        
        # 標準化所有特徵到0-100範圍
        feature_columns = ['driver_skill', 'team_skill', 'driver_consistency', 
                          'team_reliability', 'driver_circuit_expertise', 'recent_form']
        
        for col in feature_columns:
            if df[col].std() > 0:  # 避免除零
                scaler = MinMaxScaler(feature_range=(10, 90))  # 避免極端值
                df[col] = scaler.fit_transform(df[[col]]).flatten()
        
        # 計算組合特徵
        df['overall_driver_rating'] = (df['driver_skill'] * 0.4 + 
                                      df['driver_consistency'] * 0.3 + 
                                      df['recent_form'] * 0.3)
        
        df['overall_team_rating'] = (df['team_skill'] * 0.7 + 
                                    df['team_reliability'] * 0.3)
        
        df['combined_rating'] = (df['overall_driver_rating'] * 0.6 + 
                                df['overall_team_rating'] * 0.4)
        
        return df

    def build_enhanced_ranking_model(self, input_dim):
        """改進的排名預測模型"""
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(input_dim,)),
            
            # 第一層 - 特徵提取
            tf.keras.layers.Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),
            
            # 第二層 - 深度特徵學習
            tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),
            
            # 第三層 - 模式識別
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dropout(0.25),
            
            # 第四層 - 精細調整
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dropout(0.2),
            
            # 輸出層
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dense(1, activation='linear')
        ])
        
        # 使用更好的優化器設置
        optimizer = tf.keras.optimizers.Adam(
            learning_rate=0.001,
            beta_1=0.9,
            beta_2=0.999,
            epsilon=1e-7
        )
        
        model.compile(
            optimizer=optimizer,
            loss='huber',  # 對異常值更穩健
            metrics=['mae', 'mse']
        )
        return model

    def train_enhanced_models(self, df):
        """訓練增強模型"""
        print("開始訓練增強模型...")
        df_processed = self.prepare_data(df)
        
        # 準備訓練數據 - 使用更多特徵
        ranking_features = [
            'driver_encoded', 'team_encoded', 'circuit_encoded', 'year',
            'driver_skill', 'team_skill', 'driver_consistency', 'team_reliability',
            'driver_circuit_expertise', 'recent_form', 'overall_driver_rating',
            'overall_team_rating', 'combined_rating'
        ]
        
        X_ranking = df_processed[ranking_features].values
        y_ranking = df_processed['position_numeric'].values
        
        # 數據標準化
        self.scalers['ranking'] = StandardScaler()
        X_ranking_scaled = self.scalers['ranking'].fit_transform(X_ranking)
        
        # 分割數據 - 使用分層抽樣
        X_train, X_test, y_train, y_test = train_test_split(
            X_ranking_scaled, y_ranking, test_size=0.2, random_state=42, stratify=None)
        
        # 進一步分割驗證集
        X_train, X_val, y_train, y_val = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42)
        
        print(f"訓練集: {len(X_train)}, 驗證集: {len(X_val)}, 測試集: {len(X_test)}")
        
        # 建立模型
        self.ranking_model = self.build_enhanced_ranking_model(X_ranking_scaled.shape[1])
        
        # 回調函數
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=20, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6),
            tf.keras.callbacks.ModelCheckpoint(
                'best_model.h5', monitor='val_loss', save_best_only=True)
        ]
        
        # 訓練模型
        history = self.ranking_model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=150,
            batch_size=32,
            callbacks=callbacks,
            verbose=1
        )
        
        # 評估模型
        test_predictions = self.ranking_model.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_predictions)
        test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
        
        print(f"測試集 MAE: {test_mae:.2f}")
        print(f"測試集 RMSE: {test_rmse:.2f}")
        
        # 計算準確率（±2位以內）
        accuracy_2 = np.mean(np.abs(y_test - test_predictions.flatten()) <= 2) * 100
        accuracy_3 = np.mean(np.abs(y_test - test_predictions.flatten()) <= 3) * 100
        
        print(f"±2位準確率: {accuracy_2:.1f}%")
        print(f"±3位準確率: {accuracy_3:.1f}%")
        
        print("增強模型訓練完成！")
        return history

    def predict_race_ranking_enhanced(self, circuit_name, year, selected_drivers, top_n=10):
        """增強的比賽排名預測"""
        print(f"預測 {circuit_name} {year} 年比賽")
        
        results = []
        predictions = []
        
        for driver_name in selected_drivers:
            if driver_name not in self.active_drivers:
                continue
                
            current_team = self.active_drivers[driver_name]['current_team']
            
            try:
                # 編碼特徵
                driver_encoded = self.encoders['driver'].transform([driver_name])[0]
                team_encoded = self.encoders['team'].transform([current_team])[0]
                circuit_encoded = self.encoders['circuit'].transform([circuit_name])[0]
                
                # 獲取車手和車隊的歷史表現指標
                driver_info = self.active_drivers[driver_name]
                
                # 構建預測特徵向量
                features = np.array([[
                    driver_encoded, team_encoded, circuit_encoded, year,
                    driver_info.get('activity_score', 0.5) * 100,  # driver_skill
                    50 + np.random.normal(0, 10),  # team_skill (可以根據實際數據改進)
                    (1 - driver_info.get('avg_position', 15) / 20) * 100,  # driver_consistency
                    driver_info.get('finish_rate', 0.8) * 100,  # team_reliability
                    50 + np.random.normal(0, 15),  # driver_circuit_expertise
                    driver_info.get('activity_score', 0.5) * 100,  # recent_form
                    driver_info.get('activity_score', 0.5) * 100,  # overall_driver_rating
                    50 + np.random.normal(0, 10),  # overall_team_rating
                    driver_info.get('activity_score', 0.5) * 100   # combined_rating
                ]])
                
                features_scaled = self.scalers['ranking'].transform(features)
                predicted_position = self.ranking_model.predict(features_scaled, verbose=0)[0][0]
                
                # 添加一些隨機性來模擬比賽的不確定性
                uncertainty = np.random.normal(0, 1.5)
                final_prediction = max(1, predicted_position + uncertainty)
                
                result = {
                    'driver': driver_name,
                    'team': current_team,
                    'predicted_position': float(final_prediction),
                    'confidence': self._calculate_enhanced_confidence(driver_name, current_team),
                    'activity_score': driver_info.get('activity_score', 0.5),
                    'avg_position': driver_info.get('avg_position', 15),
                    'finish_rate': driver_info.get('finish_rate', 0.8)
                }
                
                results.append(result)
                predictions.append(final_prediction)
                
            except Exception as e:
                print(f"預測 {driver_name} 時發生錯誤: {e}")
                continue
        
        if not results:
            return []
        
        # 按預測排名排序
        results.sort(key=lambda x: x['predicted_position'])
        
        # 重新分配連續排名
        final_results = []
        for i, result in enumerate(results[:top_n]):
            result['final_position'] = i + 1
            final_results.append(result)
            
        return final_results

    def _calculate_enhanced_confidence(self, driver_name, team_name):
        """計算增強的預測信心度"""
        if driver_name not in self.active_drivers:
            return 0.3
            
        driver_info = self.active_drivers[driver_name]
        
        # 多因素信心度計算
        data_confidence = min(driver_info.get('total_races', 0) / 50, 1.0)
        performance_confidence = max(0, (20 - driver_info.get('avg_position', 20)) / 20)
        reliability_confidence = driver_info.get('finish_rate', 0.5)
        activity_confidence = driver_info.get('activity_score', 0.3)
        
        overall_confidence = (
            data_confidence * 0.25 +
            performance_confidence * 0.35 +
            reliability_confidence * 0.2 +
            activity_confidence * 0.2
        )
        
        return min(max(overall_confidence, 0.1), 0.95)  # 限制在10%-95%之間

# 使用範例
def create_improved_prediction_system(training_data):
    """創建改進的預測系統"""
    system = F1PredictionSystem()
    
    # 訓練模型
    training_history = system.train_enhanced_models(training_data)
    
    return system, training_history

# 預測準確性評估函數
def evaluate_prediction_accuracy(system, test_data, circuit_name, year):
    """評估預測準確性"""
    if test_data.empty:
        return None
    
    # 獲取實際結果
    actual_results = test_data[
        (test_data['Grand Prix'] == circuit_name) & 
        (test_data['year'] == year)
    ].copy()
    
    if actual_results.empty:
        return None
    
    drivers_to_predict = actual_results['Driver'].tolist()
    
    # 進行預測
    predictions = system.predict_race_ranking_enhanced(
        circuit_name, year, drivers_to_predict, len(drivers_to_predict))
    
    if not predictions:
        return None
    
    # 計算準確性指標
    pred_dict = {pred['driver']: pred['final_position'] for pred in predictions}
    actual_dict = {}
    
    for _, row in actual_results.iterrows():
        pos = pd.to_numeric(row['Pos'] if 'Pos' in row else row.get('Position', 21), errors='coerce')
        if pd.notna(pos):
            actual_dict[row['Driver']] = int(pos)
    
    # 計算各種準確性指標
    common_drivers = set(pred_dict.keys()) & set(actual_dict.keys())
    
    if not common_drivers:
        return None
    
    mae = np.mean([abs(pred_dict[driver] - actual_dict[driver]) for driver in common_drivers])
    rmse = np.sqrt(np.mean([(pred_dict[driver] - actual_dict[driver])**2 for driver in common_drivers]))
    
    # 計算不同誤差範圍內的準確率
    accuracy_1 = np.mean([abs(pred_dict[driver] - actual_dict[driver]) <= 1 for driver in common_drivers]) * 100
    accuracy_2 = np.mean([abs(pred_dict[driver] - actual_dict[driver]) <= 2 for driver in common_drivers]) * 100
    accuracy_3 = np.mean([abs(pred_dict[driver] - actual_dict[driver]) <= 3 for driver in common_drivers]) * 100
    
    return {
        'mae': mae,
        'rmse': rmse,
        'accuracy_1': accuracy_1,
        'accuracy_2': accuracy_2,
        'accuracy_3': accuracy_3,
        'drivers_compared': len(common_drivers)
    }

In [4]:
def create_gradio_interface(prediction_system):
    """創建Gradio界面"""
    
    def format_prediction_results(circuit, year, selected_drivers, top_n):
        if not selected_drivers:
            return "請選擇至少一位車手進行預測"
            
        try:
            results = prediction_system.predict_race_ranking(
                circuit, int(year), selected_drivers, int(top_n))
            
            if not results:
                return "預測失敗，請檢查選擇的車手"
            
            # 格式化輸出
            output = f"## 🏁 {circuit} {year}年 比賽預測結果 (前{len(results)}名)\n\n"
            
            output += "| 排名 | 車手 | 當前車隊 | 車手實力 | 車隊實力 | 綜合實力 | 信心度 |\n"
            output += "|------|------|----------|----------|----------|----------|--------|\n"
            
            for result in results:
                combined_skill = (result['driver_skill'] + result['team_skill']) / 2
                confidence_pct = result['confidence'] * 100
                
                output += f"| {result['final_position']} | {result['driver']} | {result['team']} | "
                output += f"{result['driver_skill']:.1f} | {result['team_skill']:.1f} | "
                output += f"{combined_skill:.1f} | {confidence_pct:.0f}% |\n"
            
            # 添加預測邏輯說明
            output += "\n\n### 📊 預測邏輯說明\n"
            output += """
**實力計算:**
- **車手實力**: 基於歷史比賽成績，使用指數移動平均突出近期表現
- **車隊實力**: 基於車隊歷年平均表現，反映技術水平和資源
- **綜合實力**: 車手實力與車隊實力的平均值

**排名預測:**
- 結合車手實力、車隊實力、賽道特性和年份因素
- 使用深度學習模型分析歷史數據模式
- 考慮車手與當前車隊的配合程度

**信心度:**
- 基於車手的歷史比賽數據量
- 考慮車手與當前車隊的合作經驗
- 數值越高表示預測越可靠
            """
            
            return output
            
        except Exception as e:
            return f"預測過程中發生錯誤: {str(e)}"

    # 獲取活躍車手列表
    active_driver_list = list(prediction_system.active_drivers.keys())
    active_driver_list.sort()
    
    with gr.Blocks(title="F1 比賽預測系統", theme=gr.themes.Soft()) as interface:
        gr.Markdown("# 🏎️ F1 比賽預測系統")
        gr.Markdown("""
        ### 🎯 系統特色
        - **智能車隊匹配**: 自動為車手匹配當前車隊，無需手動選擇
        - **活躍車手篩選**: 只顯示近期活躍的車手，確保預測相關性  
        - **靈活排名輸出**: 可自定義顯示前幾名的預測結果
        - **透明預測邏輯**: 詳細說明實力計算和排名預測方法
        """)
        
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### ⚙️ 比賽設定")
                
                circuit_dropdown = gr.Dropdown(
                    choices=prediction_system.circuit_names,
                    label="🏁 選擇賽道",
                    value=prediction_system.circuit_names[0] if prediction_system.circuit_names else None
                )
                
                year_slider = gr.Slider(
                    minimum=2020,
                    maximum=2030,
                    step=1,
                    value=2024,
                    label="📅 比賽年份"
                )
                
                drivers_dropdown = gr.Dropdown(
                    choices=active_driver_list,
                    label="🏎️ 選擇參賽車手",
                    multiselect=True,
                    value=active_driver_list[:10] if len(active_driver_list) >= 10 else active_driver_list,
                    info=f"從 {len(active_driver_list)} 位活躍車手中選擇"
                )
                select_all_btn = gr.Button("全選/取消全選")

                # 全選/取消全選 callback
                def toggle_select_all_drivers(selected, choices):
                    if selected and set(selected) == set(choices):
                        # 如果已經全選→改成全取消
                        return []
                    return choices  # 否則就全選

                select_all_btn.click(
                    fn=toggle_select_all_drivers,
                    inputs=[drivers_dropdown, gr.State(active_driver_list)],
                    outputs=drivers_dropdown
                )
                
                top_n_slider = gr.Slider(
                    minimum=3,
                    maximum=20,
                    step=1,
                    value=10,
                    label="🏆 顯示排名數量",
                    info="選擇要顯示的前N名結果"
                )
                
                predict_button = gr.Button("🚀 開始預測", variant="primary", size="lg")
                
                # 系統統計
                gr.Markdown(f"""
                **📈 系統數據:**
                - 活躍車手: {len(active_driver_list)} 位
                - 可選賽道: {len(prediction_system.circuit_names)} 個
                - 訓練數據: {len(prediction_system.training_data) if prediction_system.training_data is not None else 0} 筆記錄
                """)
                
            with gr.Column(scale=2):
                gr.Markdown("### 🏆 預測結果")
                results_output = gr.Markdown(
                    value="選擇車手和比賽參數，然後點擊「開始預測」查看結果",
                    elem_classes=["prediction-output"]
                )
        
        predict_button.click(
            fn=format_prediction_results,
            inputs=[circuit_dropdown, year_slider, drivers_dropdown, top_n_slider],
            outputs=[results_output]
        )
        
        # 活躍車手信息展示
        with gr.Row():
            gr.Markdown("### 👥 活躍車手一覽")
            
        active_drivers_info = "| 車手 | 當前車隊 | 總比賽數 | 最後活躍年 |\n|------|----------|----------|------------|\n"
        for driver, info in list(prediction_system.active_drivers.items())[:15]:
            active_drivers_info += f"| {driver} | {info['current_team']} | {info['total_races']} | {info['last_active_year']} |\n"
        
        if len(prediction_system.active_drivers) > 15:
            active_drivers_info += f"\n*還有 {len(prediction_system.active_drivers) - 15} 位活躍車手...*"
            
        gr.Markdown(active_drivers_info)
    
    return interface

In [5]:
def main():
    print("載入F1數據...")
    try:
        df = pd.read_csv('./f1_merged_all.csv')
        print(f"成功載入數據: {df.shape}")
    except FileNotFoundError:
        print("未找到 f1_merged_all.csv，創建示例數據...")
        np.random.seed(42)
        drivers = ['Hamilton', 'Verstappen', 'Leclerc', 'Russell', 'Sainz', 'Norris', 'Piastri', 'Alonso']
        teams = ['Mercedes', 'Red Bull', 'Ferrari', 'McLaren', 'Aston Martin']
        circuits = ['Monaco', 'Silverstone', 'Monza', 'Spa', 'Suzuka']
        years = [2020, 2021, 2022, 2023, 2024]
        data = []
        for year in years:
            for circuit in circuits:
                for i, driver in enumerate(drivers):
                    team = teams[i % len(teams)]
                    position = np.random.randint(1, 21)
                    data.append({
                        'Driver': driver,
                        'Team': team,
                        'Grand Prix': circuit,
                        'year': year,
                        'Position': position,
                        'Fastest Lap Time': f"1:{np.random.randint(15, 45)}.{np.random.randint(100, 999)}"
                    })
        df = pd.DataFrame(data)
        df.to_csv('f1_demo_data.csv', index=False)
        print("創建了示例數據檔案: f1_demo_data.csv")
    prediction_system = F1PredictionSystem()
    prediction_system.train_models(df)
    interface = create_gradio_interface(prediction_system)
    print("啟動Gradio界面...")
    interface.launch()

if __name__ == '__main__':
    main()


載入F1數據...
成功載入數據: (2284, 12)


AttributeError: 'F1PredictionSystem' object has no attribute 'train_models'